In [35]:
# load the clean dataset
import pandas as pd

df_clean = pd.read_parquet(r"C:\Users\zeid9\Documents\Scalable-MLOps-Pipeline-for-Credit-Risk-Prediction\data\df_clean.parquet")

In [36]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   age                  32581 non-null  int64  
 1   income               32581 non-null  int64  
 2   home_status          32581 non-null  object 
 3   emp_years            32581 non-null  float64
 4   loan_intent          32581 non-null  object 
 5   loan_grade           32581 non-null  object 
 6   loan_amount          32581 non-null  int64  
 7   loan_int_rate        32581 non-null  float64
 8   loan_status          32581 non-null  int64  
 9   loan_percent_income  32581 non-null  float64
 10  default              32581 non-null  object 
 11  credit_history       32581 non-null  int64  
dtypes: float64(3), int64(5), object(4)
memory usage: 3.0+ MB


In [37]:
# Map 'Y' to 1 and 'N' to 0 in the 'default' column
df_clean['default'] = df_clean['default'].map({'Y': 1, 'N': 0})

In [39]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression

# Define feature columns
cat_cols = ['home_status', 'loan_intent', 'loan_grade']
num_cols = ['income', 'emp_years', 'loan_amount',
            'loan_int_rate', 'loan_percent_income',
            'credit_history']

# Define input and target
X = df_clean[cat_cols + num_cols]
y = df_clean['default']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Define preprocessing for columns
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols),
    ('num', StandardScaler(), num_cols)
])

# define the model
from sklearn.linear_model import LogisticRegression
lg = LogisticRegression(class_weight= 'balanced', random_state=42)

# Define full pipeline with preprocessing, SMOTE, and model
lg_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', lg)
])

# Fit the pipeline on training data
lg_pipeline.fit(X_train, y_train)

# Make predictions
y_pred = lg_pipeline.predict(X_test)


In [40]:
from sklearn.metrics import precision_recall_curve, f1_score, accuracy_score, confusion_matrix, roc_auc_score
# Accuracy
print("Accuracy Score:", accuracy_score(y_test, y_pred))

# Evaluate
print(classification_report(y_test, y_pred))

# Confusion matrix
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# ROC AUC Score
print("ROC AUC Score:", roc_auc_score(y_test, y_pred))

Accuracy Score: 0.8276814485192573
              precision    recall  f1-score   support

           0       1.00      0.79      0.88      5368
           1       0.51      1.00      0.67      1149

    accuracy                           0.83      6517
   macro avg       0.75      0.90      0.78      6517
weighted avg       0.91      0.83      0.85      6517


Confusion Matrix:
 [[4245 1123]
 [   0 1149]]
ROC AUC Score: 0.8953986587183309


## Tunning the model using GridSearchCV

In [41]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, RandomizedSearchCV
import numpy as np
import time

# Define hyperparameter grid
param_grid = {
    'classifier__C': [0.01, 0.1, 1, 10],
    'classifier__class_weight': ['balanced', None],
    'classifier__penalty': ['l2'],
    'classifier__solver': ['liblinear', 'saga']
}

# Grid Search
grid = RandomizedSearchCV(
    estimator=lg_pipeline,
    param_distributions=param_grid, # param_grid for GSCV and parameter_distribution for RSCV
    cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
    n_iter=100, # only work RSCV
    scoring='recall', 
    n_jobs=-2,
    verbose=2,
    random_state=42 # only work RSCV
)
# ETA calculation
start = time.time()
# Fit on the original X_train and y_train (not resampled)
grid.fit(X_train, y_train)

print(f"\nTraining completed in {(time.time() - start)/60:.2f} minutes.")


c:\Users\zeid9\Documents\Scalable-MLOps-Pipeline-for-Credit-Risk-Prediction\venv\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 16 is smaller than n_iter=100. Running 16 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 10 folds for each of 16 candidates, totalling 160 fits

Training completed in 0.85 minutes.


In [42]:
print("The best parameters are %s with a score of %0.2f"
      % (grid.best_params_, grid.best_score_))

The best parameters are {'classifier__solver': 'liblinear', 'classifier__penalty': 'l2', 'classifier__class_weight': 'balanced', 'classifier__C': 1} with a score of 1.00


In [43]:

import pandas as pd

grid_results = pd.concat(
    [pd.DataFrame(grid.cv_results_["params"]),
     pd.DataFrame(grid.cv_results_["mean_test_score"], 
                  columns=["Recall"])],axis=1)
grid_results

,classifier__solver,classifier__penalty,classifier__class_weight,classifier__C,Recall
0,liblinear,l2,balanced,0.01,0.980200
1,saga,l2,balanced,0.01,0.977587
2,liblinear,l2,None,0.01,0.980200
3,saga,l2,None,0.01,0.977587
4,liblinear,l2,balanced,0.10,0.997171
5,saga,l2,balanced,0.10,0.988249
6,liblinear,l2,None,0.10,0.997171
7,saga,l2,None,0.10,0.988249
8,liblinear,l2,balanced,1.00,1.000000
9,saga,l2,balanced,1.00,0.988032


In [ ]:
selected_grid_var =grid_results[['classifier__max_features', 'classifier__n_estimators', 'Recall']]
grid_contour = selected_grid_var.groupby(
    ['classifier__max_features', 'classifier__n_estimators']
).mean()

grid_contour_expanded = grid_contour.unstack(level='classifier__n_estimators')
grid_contour_expanded

KeyError: "['classifier__max_features', 'classifier__n_estimators'] not in index"

In [ ]:

x = grid_contour_expanded.columns.levels[1].values
y = grid_contour_expanded.index.values
z = grid_contour_expanded.values

In [ ]:

import plotly.graph_objects as go

# x = n_estimators (columns)
# y = max_features (rows)
# z = Recall

fig = go.Figure(data=[go.Surface(z=z, x=x, y=y)])

fig.update_layout(
    title='Hyperparameter Tuning: Accuracy by n_estimators & max_features',
    scene=dict(
        xaxis=dict(title='n_estimators'),
        yaxis=dict(title='max_features'),
        zaxis=dict(title='Recall'),
    ),
    autosize=False,
    width=800,
    height=800,
    margin=dict(l=65, r=50, b=65, t=90)
)

fig.show()


# Model after tuning

In [45]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Get the best model from RandomizedSearchCV
tuned_lg_pipeline = grid.best_estimator_

# Predict on the test set
y_pred = tuned_lg_pipeline.predict(X_test)

# Evaluate the model
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("ROC AUC Score:", roc_auc_score(y_test, y_pred))


Confusion Matrix:
 [[4245 1123]
 [   0 1149]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.79      0.88      5368
           1       0.51      1.00      0.67      1149

    accuracy                           0.83      6517
   macro avg       0.75      0.90      0.78      6517
weighted avg       0.91      0.83      0.85      6517

ROC AUC Score: 0.8953986587183309


In [46]:
import joblib
from datetime import datetime
from sklearn.metrics import recall_score

# Define your model version and recall score
model_version = 1
recall_score = recall_score(y_test, tuned_lg_pipeline.predict(X_test))
model_type = 'lg'    
today = datetime.today().strftime('%Y-%m-%d')

# Format the filename
filename = f"{model_type}_v{model_version}_recall{int(recall_score * 100)}_{today}.pkl"

# Define save directory (you can customize the folder)
save_path = f"C:/Users/zeid9/Documents/Scalable-MLOps-Pipeline-for-Credit-Risk-Prediction/models/{filename}"

# Save the model
joblib.dump(tuned_lg_pipeline, save_path)

print(f"Model saved to: {save_path}")


Model saved to: C:/Users/zeid9/Documents/Scalable-MLOps-Pipeline-for-Credit-Risk-Prediction/models/lg_v1_recall100_2025-08-14.pkl


In [ ]:
# Example input sample in original form (dict or DataFrame)
sample = {
    'income': 60000,
    'home_status': 'RENT',
    'emp_years': 4,
    'loan_intent': 'MEDICAL',
    'loan_grade': 'D',
    'loan_amount': 10000,
    'loan_int_rate': 12.14,
    'loan_percent_income': 12.8,
    'credit_history': 6
}

import pandas as pd
input_df = pd.DataFrame([sample])

# Get predicted probabilities
probs = tuned_lg_pipeline.predict_proba(input_df)

# Manually set your threshold
threshold = 0.68  # 

# Get the probability for class 1 (positive class)
prob_class_1 = probs[:, 1]

# Apply threshold
custom_prediction = (prob_class_1 > threshold).astype(int)

# Print result
print(f"Default Threshold ({threshold}) Prediction:", custom_prediction[0])
print(f"Probability of Default (class 1): {prob_class_1[0] * 100:.2f}%")

# Interpretive message
if custom_prediction[0] == 1:
    print("Prediction: YES - Likely to Default")
else:
    print("Prediction: NO - Unlikely to Default")


ValueError: columns are missing: {'loan_grade'}